In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
import openpyxl
from PIL import Image, ImageDraw, ImageFont

import os

In [2]:
path_training = '../data/MGAMtC_updated_chemical_list.xlsx'

In [3]:
df_training = pd.read_excel(path_training)

In [4]:
df_training["Fingerprint"] = [
    np.array(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, nBits=2048))
    for s in df_training["SMILES"]
]

[11:56:15] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerator
[11:56:16] DEPRECATION WARNING: please use MorganGenerat

In [5]:
def draw_bit_for_molecules(df, bit_of_interest, radius=2, nBits=2048):

    output_dir = f'../Figures/Bits_NOT_BINDING/bit_{bit_of_interest}'
    os.makedirs(output_dir, exist_ok=True)
    
    binding_df = df[df['Binding'] == 'Yes'].copy()
    
    molecules_with_bit = 0
    
    for idx, row in binding_df.iterrows():
        smiles = row['SMILES']
        chem_name = row['Chemical Name']
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Warning: Could not parse SMILES for {chem_name}")
            continue
        
        bitInfo = {}
        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits, bitInfo=bitInfo)
        
        if bit_of_interest in bitInfo:
            molecules_with_bit += 1
            occurrences = bitInfo[bit_of_interest]
            
            for occ_idx, (atom_idx, rad) in enumerate(occurrences):
                bond_ids = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
                
                atom_ids = set([atom_idx])
                for bidx in bond_ids:
                    bond = mol.GetBondWithIdx(bidx)
                    atom_ids.add(bond.GetBeginAtomIdx())
                    atom_ids.add(bond.GetEndAtomIdx())
                
                drawer = rdMolDraw2D.MolDraw2DSVG(600, 500)
                drawer.drawOptions().addAtomIndices = False
                
                highlight_atoms = {aid: (1, 0.7, 0.7) for aid in atom_ids} 
                highlight_bonds = {bid: (1, 0.3, 0.3) for bid in bond_ids}  
                highlight_radii = {aid: 0.5 for aid in atom_ids}  
                
                title = f"{chem_name} - Bit Location {bit_of_interest}"
                
                drawer.DrawMolecule(
                    mol,
                    highlightAtoms=list(atom_ids),
                    highlightAtomColors=highlight_atoms,
                    highlightBonds=list(bond_ids),
                    highlightBondColors=highlight_bonds,
                    highlightAtomRadii=highlight_radii,
                    legend=title
                )
                
                drawer.FinishDrawing()
                svg = drawer.GetDrawingText()
                
                safe_name = "".join(c for c in chem_name if c.isalnum() or c in (' ', '-', '_')).strip()
                safe_name = safe_name.replace(' ', '_')
                
                if len(occurrences) > 1:
                    filename = f"{safe_name}_occurrence_{occ_idx+1}.svg"
                else:
                    filename = f"{safe_name}.svg"
                
                filepath = os.path.join(output_dir, filename)
                
                with open(filepath, 'w') as f:
                    f.write(svg)
                
                print(f"Saved: {filename}")
                print(f"  Center atom {atom_idx}, Radius {rad}")
                print(f"  Atoms: {sorted(atom_ids)}, Bonds: {sorted(bond_ids)}\n")
    
    print(f"\nSummary:")
    print(f"Total molecules with Binding='Yes': {len(binding_df)}")
    print(f"Molecules with bit {bit_of_interest} active: {molecules_with_bit}")
    print(f"Figures saved to: {output_dir}")


In [6]:
bit_of_interest = 1380                      #change this based on intested bit
draw_bit_for_molecules(df_training, bit_of_interest)


Saved: 1-Deoxy-1-morpholino-D-fructose.svg
  Center atom 14, Radius 1
  Atoms: [4, 13, 14, 15], Bonds: [13, 14, 17]

Saved: D---Tagatose.svg
  Center atom 3, Radius 1
  Atoms: [2, 3, 4, 9], Bonds: [2, 3, 8]

Saved: D--Sorbose.svg
  Center atom 3, Radius 1
  Atoms: [2, 3, 4, 9], Bonds: [2, 3, 8]

Saved: D-Psicose.svg
  Center atom 3, Radius 1
  Atoms: [2, 3, 4, 9], Bonds: [2, 3, 8]

Saved: Voglibose.svg
  Center atom 4, Radius 1
  Atoms: [3, 4, 5, 9], Bonds: [3, 4, 8]

Saved: Acarbose.svg
  Center atom 35, Radius 0
  Atoms: [35], Bonds: []


Summary:
Total molecules with Binding='Yes': 27
Molecules with bit 1380 active: 6
Figures saved to: ../Figures/Bits_NOT_BINDING/bit_1380


[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerator
[11:57:14] DEPRECATION WARNING: please use MorganGenerat

### Draw individual bit locations

In [7]:
def draw_bit_substructure(df, bit_of_interest, radius=2, nBits=2048):

    output_dir = '../Figures/Bit_Locations'
    os.makedirs(output_dir, exist_ok=True)
    
    binding_df = df[df['Binding'] == 'Yes'].copy()
    
    for idx, row in binding_df.iterrows():
        smiles = row['SMILES']
        chem_name = row['Chemical Name']
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        
        bitInfo = {}
        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits, bitInfo=bitInfo)
        
        if bit_of_interest in bitInfo:
            occurrences = bitInfo[bit_of_interest]
            atom_idx, rad = occurrences[0]  #take the first occurence
            
            #get substructure env
            bond_ids = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
            
            atom_ids = set([atom_idx])
            for bidx in bond_ids:
                bond = mol.GetBondWithIdx(bidx)
                atom_ids.add(bond.GetBeginAtomIdx())
                atom_ids.add(bond.GetEndAtomIdx())
            
            #extract as SMILES and and rhen convert to mol to ensure we only have the substructure
            fragment_smiles = Chem.MolFragmentToSmiles(mol, list(atom_ids), bondsToUse=list(bond_ids))
            fragment_mol = Chem.MolFromSmiles(fragment_smiles)
            
            if fragment_mol is None:
                print(f"Could not create fragment molecule for bit {bit_of_interest}")
                continue
            
            drawer = rdMolDraw2D.MolDraw2DSVG(400, 400)
            drawer.drawOptions().addAtomIndices = False
            
            title = f"Bit {bit_of_interest} Substructure"
            drawer.DrawMolecule(fragment_mol, legend=title)
            drawer.FinishDrawing()
            svg = drawer.GetDrawingText()
            
            filepath = os.path.join(output_dir, f"bit_{bit_of_interest}_substructure.svg")
            
            with open(filepath, 'w') as f:
                f.write(svg)
            
            print(f"Bit {bit_of_interest} substructure saved to: {filepath}")
            print(f"Fragment SMILES: {fragment_smiles}")
            print(f"Extracted from molecule: {chem_name}")
            return filepath
    
    print(f"Bit {bit_of_interest} not found in any binding molecules")
    return None

In [8]:
bit = 1516                      #change this based on interested bit
draw_bit_substructure(df_training, bit)

Bit 1516 substructure saved to: ../Figures/Bit_Locations/bit_1516_substructure.svg
Fragment SMILES: CCO
Extracted from molecule: 2-hydroxymethyl-tetrahydro-pyran-3,4,5-triol


[11:59:51] DEPRECATION WARNING: please use MorganGenerator
[11:59:51] DEPRECATION WARNING: please use MorganGenerator


'../Figures/Bit_Locations/bit_1516_substructure.svg'